# Load the NWM v3.1 streamflow backfill (#439)

Loads the parquet written by `fetching/fetch_nwm31_streamflow_backfill.ipynb` into
`secondary_timeseries`. The `nwm31_*` configurations were added by
`22_load_nwm31_domain_variables.ipynb`.

Rows the scheduled `nwm31` ingest already wrote are skipped (`write_mode="append"`,
`drop_duplicates=True`).

In [ ]:
from pathlib import Path

import teehr
from teehr.evaluation.spark_session_utils import create_spark_session
from teehr.fetching.utils import format_nwm_configuration_metadata
from pyspark.sql import functions as F

teehr.__version__

In [ ]:
NWM_VERSION = "nwm31"
CUTOVER = "2026-08-18 00:00:00"
PARQUET_DIR = Path("/data/data_processing/backfill-nwm31/parquet")

CONFIGS = [
    ("short_range", "channel_rt"),
    ("short_range_alaska", "channel_rt"),
    ("short_range_hawaii", "channel_rt"),
    ("short_range_puertorico", "channel_rt"),
    ("medium_range_mem1", "channel_rt_1"),
    ("medium_range_blend", "channel_rt"),
    ("medium_range_alaska_mem1", "channel_rt_1"),
    ("medium_range_blend_alaska", "channel_rt"),
    ("analysis_assim_no_da", "channel_rt"),
    ("analysis_assim_alaska_no_da", "channel_rt"),
    ("analysis_assim_extend_no_da", "channel_rt"),
    ("analysis_assim_extend_alaska_no_da", "channel_rt"),
    ("analysis_assim_hawaii_no_da", "channel_rt"),
    ("analysis_assim_puertorico_no_da", "channel_rt"),
]

About a month per configuration (CONUS short range is roughly 200-250M rows), so this uses
the Prefect flows' default cluster shape: 2 executors per r5.4xlarge node. For a
configuration above about 20-30 GB of parquet, raise `executor_instances` to 6-8.

In [ ]:
%%time
spark = create_spark_session(
    start_spark_cluster=True,
    executor_instances=4,
    executor_memory="50g",
    executor_cores=7,
    update_configs={
        "spark.sql.shuffle.partitions": "64",
    },
)

In [ ]:
ev = teehr.RemoteReadWriteEvaluation(spark=spark)

Check that every `nwm31_*` configuration exists, and list what was fetched.

In [ ]:
config_names = [
    format_nwm_configuration_metadata(c, NWM_VERSION)["name"] for c, _ in CONFIGS
]
existing = {
    row["name"] for row in ev.configurations.to_sdf().select("name").collect()
}
print(f"Missing configurations: {sorted(set(config_names) - existing)}")

cache_dirs = {}
for name in config_names:
    dirs = [d for d in Path(PARQUET_DIR, name).glob("*") if d.is_dir()]
    for d in dirs:
        files = list(d.glob("*.parquet"))
        size = sum(f.stat().st_size for f in files) / 1024**3
        print(f"{name}/{d.name}: {len(files)} files, {size:.2f} GB")
    if dirs:
        cache_dirs[name] = dirs

#### Load one configuration

In [ ]:
config_name = "nwm31_short_range"
cache_dir = cache_dirs[config_name][0]
cache_dir

In [ ]:
nwm_sdf = ev.spark.read.parquet(cache_dir.as_posix())
nwm_sdf.count()

In [ ]:
nwm_sdf.select(
    F.min("reference_time"), F.max("reference_time"),
    F.min("value_time"), F.max("value_time"),
    F.countDistinct("location_id"),
).show(truncate=False)

In [ ]:
nwm_sdf.show(n=5, truncate=False)

In [ ]:
%%time
ev.secondary_timeseries.load_dataframe(
    nwm_sdf,
    write_mode="append",
    drop_duplicates=True,
)

#### Check for a gap at either end
Runs across every configuration, so it can be re-run after each load. For forecasts, the
earliest `reference_time` should be the cutover. For analysis configs `reference_time` is
NULL, so check `value_time` instead.

In [ ]:
%%time
names_sql = ", ".join(f"'{n}'" for n in config_names)
ev.spark.sql(f"""
    SELECT
        configuration_name,
        MIN(reference_time) AS min_reference_time,
        MAX(reference_time) AS max_reference_time,
        MIN(value_time) AS min_value_time,
        MAX(value_time) AS max_value_time,
        COUNT(DISTINCT reference_time) AS n_reference_times
    FROM iceberg.teehr.secondary_timeseries
    WHERE configuration_name IN ({names_sql})
    GROUP BY configuration_name
    ORDER BY configuration_name
""").toPandas()

In [ ]:
spark.stop()

#### NOTE: Once every configuration is loaded and checked, delete the local data at:

In [ ]:
PARQUET_DIR.parent